# Задание 2. Определение присутствия людей в помещении по показаниям датчиков CO₂, температуры, влажности и освещённости

**Постановка задачи**

| | |
|---|---|
| **Входные данные** | показания датчиков раз в минуту: температура (°C), относительная влажность (%), освещённость (лк), CO₂ (ppm), абсолютная влажность (кг/кг) и время измерения |
| **Что определяет модель** | есть ли в помещении люди (1) или нет (0) — бинарная классификация |
| **Данные для оценки** | открытый датасет [UCI Occupancy Detection](https://archive.ics.uci.edu/dataset/357/occupancy+detection) (Candanedo, Feldheim, 2016): офисное помещение, февраль 2015, разметка по фотографиям с камеры. Обучение — `datatraining.txt`, проверка — два отдельных периода `datatest.txt` (дверь закрыта) и `datatest2.txt` (дверь открыта) |
| **Метрики** | Accuracy, Precision, Recall, F1, ROC-AUC |

**Связь с большими данными.** Датчики в зданиях пишут показания непрерывно (здесь — 1 запись в минуту на комнату, ≈ 525 тыс. в год). Автоматическое определение занятости используется для управления вентиляцией и отоплением: по этим данным система сама решает, когда проветривать и где можно экономить энергию.

**План:** загрузка → разведочный анализ → признаки → три модели (логистическая регрессия, Random Forest, градиентный бустинг) → сравнение → важность признаков → эксперимент «только CO₂, без датчика света».

In [ ]:
# В Google Colab все библиотеки уже установлены. Локально: pip install -r requirements.txt
import io
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

## 1. Загрузка данных

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
FILES = ["datatraining.txt", "datatest.txt", "datatest2.txt"]

UCI_ZIP = "https://archive.ics.uci.edu/static/public/357/occupancy+detection.zip"
GITHUB_RAW = "https://raw.githubusercontent.com/LuisM78/Occupancy-detection-data/master/"


def download():
    # Скачивает датасет с UCI, при неудаче — с GitHub автора датасета.
    if all((DATA_DIR / f).exists() for f in FILES):
        return
    try:
        raw = urllib.request.urlopen(UCI_ZIP, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            for name in z.namelist():
                if Path(name).name in FILES:
                    (DATA_DIR / Path(name).name).write_bytes(z.read(name))
        print("Скачано с UCI")
    except Exception as e:
        print(f"UCI недоступен ({e}), качаю с GitHub")
        for f in FILES:
            urllib.request.urlretrieve(GITHUB_RAW + f, DATA_DIR / f)
    missing = [f for f in FILES if not (DATA_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Не удалось скачать {missing}. Скачайте архив вручную с "
            "https://archive.ics.uci.edu/dataset/357/occupancy+detection и распакуйте в папку data/"
        )


def load(name):
    df = pd.read_csv(DATA_DIR / name, index_col=0, parse_dates=["date"])
    return df.sort_values("date").reset_index(drop=True)


download()
train_raw, test1_raw, test2_raw = (load(f) for f in FILES)

for name, df in [("train", train_raw), ("test1 (дверь закрыта)", test1_raw), ("test2 (дверь открыта)", test2_raw)]:
    print(f"{name:24s} {len(df):6d} строк, {df['date'].min():%d.%m %H:%M} – {df['date'].max():%d.%m %H:%M}, "
          f"доля «занято» = {df['Occupancy'].mean():.1%}")
train_raw.head()

## 2. Разведочный анализ

In [ ]:
train_raw.describe().T.round(3)

In [ ]:
# Как ведут себя датчики во времени: серые полосы — в комнате есть люди
fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
for ax, col, unit in zip(axes, ["CO2", "Light", "Temperature", "Humidity"], ["ppm", "лк", "°C", "%"]):
    ax.plot(train_raw["date"], train_raw[col], lw=0.8, color="#1f5fa8")
    ax.fill_between(train_raw["date"], 0, 1, where=train_raw["Occupancy"] == 1,
                    transform=ax.get_xaxis_transform(), color="grey", alpha=0.25, lw=0)
    ax.set_ylabel(f"{col}, {unit}")
axes[0].set_title("Обучающий период: показания датчиков и занятость помещения")
fig.tight_layout()
plt.show()

In [ ]:
# Распределение признаков в двух классах
features_raw = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, col in zip(axes, features_raw):
    sns.kdeplot(data=train_raw, x=col, hue="Occupancy", common_norm=False, fill=True,
                palette={0: "#9aa0a6", 1: "#d6812d"}, ax=ax, warn_singular=False)
    ax.set_ylabel("")
fig.suptitle("Плотность распределения: 0 — пусто, 1 — есть люди", y=1.03)
fig.tight_layout()
plt.show()

plt.figure(figsize=(6.5, 5))
sns.heatmap(train_raw[features_raw + ["Occupancy"]].corr(), annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Корреляция признаков")
plt.show()

## 3. Признаки

К исходным показаниям добавим признаки, которые учитывают время и динамику:

- `hour`, `is_weekend` — люди бывают в офисе в рабочие часы и будни;
- `CO2_diff5` — изменение CO₂ за 5 минут: при входе людей концентрация растёт, после ухода падает;
- `Light_diff5` — изменение освещённости за 5 минут (включили или выключили свет).

Разности считаются отдельно внутри каждого файла, так что информация между обучающей и тестовой выборками не перетекает.

In [ ]:
def add_features(df):
    df = df.copy()
    df["hour"] = df["date"].dt.hour + df["date"].dt.minute / 60
    df["is_weekend"] = (df["date"].dt.dayofweek >= 5).astype(int)
    df["CO2_diff5"] = df["CO2"].diff(5).fillna(0)
    df["Light_diff5"] = df["Light"].diff(5).fillna(0)
    return df


train, test1, test2 = (add_features(d) for d in (train_raw, test1_raw, test2_raw))
FEATURES = features_raw + ["hour", "is_weekend", "CO2_diff5", "Light_diff5"]
TARGET = "Occupancy"

X_train, y_train = train[FEATURES], train[TARGET]
tests = {"test1 (дверь закрыта)": test1, "test2 (дверь открыта)": test2}
print("Признаки:", FEATURES)

## 4. Обучение и сравнение моделей

In [ ]:
def make_models():
    return {
        "Логистическая регрессия": make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        "Random Forest": RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
        "Градиентный бустинг": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
    }


def evaluate(models, features):
    rows = []
    for model_name, model in models.items():
        model.fit(train[features], y_train)
        for test_name, df in tests.items():
            proba = model.predict_proba(df[features])[:, 1]
            pred = (proba >= 0.5).astype(int)
            y = df[TARGET]
            rows.append({
                "Модель": model_name, "Выборка": test_name,
                "Accuracy": accuracy_score(y, pred), "Precision": precision_score(y, pred, zero_division=0),
                "Recall": recall_score(y, pred), "F1": f1_score(y, pred), "ROC-AUC": roc_auc_score(y, proba),
            })
    return pd.DataFrame(rows)


models = make_models()
results = evaluate(models, FEATURES)
results.style.format(precision=3, subset=["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]) \
    .background_gradient(cmap="Greens", subset=["F1", "ROC-AUC"])

In [ ]:
# ROC-кривые на объединённой тестовой выборке
test_all = pd.concat(tests.values(), ignore_index=True)
fig, ax = plt.subplots(figsize=(6, 5))
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, test_all[FEATURES], test_all[TARGET], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_title("ROC-кривые (test1 + test2)")
plt.show()

best_name = results.groupby("Модель")["F1"].mean().idxmax()
best = models[best_name]
print(f"Лучшая модель по среднему F1: {best_name}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (test_name, df) in zip(axes, tests.items()):
    ConfusionMatrixDisplay.from_estimator(best, df[FEATURES], df[TARGET], display_labels=["пусто", "есть люди"],
                                          cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(test_name)
fig.suptitle(f"Матрицы ошибок — {best_name}")
fig.tight_layout()
plt.show()

## 5. Какие признаки важнее всего

In [ ]:
# Permutation importance: насколько падает F1, если перемешать значения признака
imp = permutation_importance(best, test_all[FEATURES], test_all[TARGET], scoring="f1",
                             n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp_df = pd.DataFrame({"Признак": FEATURES, "Падение F1": imp.importances_mean}) \
    .sort_values("Падение F1", ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(data=imp_df, x="Падение F1", y="Признак", color="#1f5fa8")
plt.title(f"Важность признаков ({best_name})")
plt.show()
imp_df.round(4)

## 6. Эксперимент: можно ли обойтись без датчика освещённости?

Свет — очень сильный признак, но у него есть недостаток: днём комната может быть освещена солнцем, а люди могут работать с выключенным светом. Многие недорогие устройства мониторинга воздуха меряют только CO₂, температуру и влажность. Проверим, насколько хуже работает модель без освещённости.

In [ ]:
FEATURES_NO_LIGHT = [f for f in FEATURES if not f.startswith("Light")]
results_no_light = evaluate(make_models(), FEATURES_NO_LIGHT)

compare = (
    pd.concat([results.assign(Признаки="все"), results_no_light.assign(Признаки="без освещённости")])
    .groupby(["Модель", "Признаки"])[["Accuracy", "F1", "ROC-AUC"]].mean()
    .unstack("Признаки")
)
compare.round(3)

In [ ]:
f1_all = results.groupby("Модель")["F1"].mean()
f1_nl = results_no_light.groupby("Модель")["F1"].mean()
x = np.arange(len(f1_all))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, f1_all.values, 0.4, label="все признаки", color="#1f5fa8")
plt.bar(x + 0.2, f1_nl[f1_all.index].values, 0.4, label="без освещённости", color="#d6812d")
plt.xticks(x, f1_all.index)
plt.ylim(min(f1_nl.min(), f1_all.min()) - 0.05, 1.0)
plt.ylabel("F1 (среднее по двум тестам)")
plt.legend()
plt.title("Вклад датчика освещённости")
plt.show()

## 7. Выводы

In [ ]:
# Итоговые цифры подставляются автоматически из результатов выше
mean_res = results.groupby("Модель")[["Accuracy", "F1", "ROC-AUC"]].mean()
b = mean_res.loc[best_name]
b_nl_name = f1_nl.idxmax()
print(f"1. Лучшая модель — {best_name}: средние Accuracy = {b['Accuracy']:.3f}, "
      f"F1 = {b['F1']:.3f}, ROC-AUC = {b['ROC-AUC']:.3f} на двух независимых тестовых периодах.")
print(f"2. Самый важный признак — {imp_df.iloc[0]['Признак']}, затем {imp_df.iloc[1]['Признак']} "
      f"и {imp_df.iloc[2]['Признак']}.")
print(f"3. Без датчика освещённости лучшая модель — {b_nl_name} с F1 = {f1_nl.max():.3f} "
      f"(со всеми признаками {f1_all.max():.3f}).")
f1_by_test = results.pivot(index="Модель", columns="Выборка", values="F1")
t1, t2 = tests
n_better = int((f1_by_test[t2] > f1_by_test[t1]).sum())
print(f"4. F1 на test2 выше, чем на test1, у {n_better} из {len(f1_by_test)} моделей:")
print(f1_by_test[[t1, t2]].round(3).to_string())
weekend_days = {name: df.loc[df["is_weekend"] == 1, "date"].dt.date.nunique()
                for name, df in [("train", train), ("test1", test1), ("test2", test2)]}
wk_rank = imp_df["Признак"].tolist().index("is_weekend") + 1
print(f"5. Признак is_weekend — на {wk_rank}-м месте по важности, хотя выходных дней в данных мало: "
      + ", ".join(f"{k} — {v}" for k, v in weekend_days.items()) + ".")

**Интерпретация.**

- Задача хорошо решается даже простыми моделями: присутствие людей сильно меняет освещённость и концентрацию CO₂. Логистическая регрессия не уступает Random Forest и градиентному бустингу, а здесь даже обходит их (п. 1): освещённость почти однозначно разделяет классы (свет включён или выключен), и более сложные модели выигрыша не дают.
- Сравнение test1 и test2 (п. 4): можно было ожидать, что при открытой двери (test2) воздух обменивается с коридором и признаки на основе CO₂ работают хуже. Но на test2 все модели показывают более высокий F1. Вероятная причина — состав периодов: test2 длиннее (неделя против двух с половиной дней) и включает ночи и выходные, когда комната пуста и ошибиться трудно, а test1 — рабочие дни, где доля «занято» выше и чаще встречаются трудные моменты прихода и ухода людей. Кроме того, главный признак — освещённость, а на неё открытая дверь не влияет.
- Второй по важности признак — `is_weekend` (п. 2, 5), и к этому стоит относиться осторожно: в обучающей выборке всего одни выходные, в test1 выходных нет совсем. Модель запомнила, что в выходные этот офис пуст. Для данного офиса это верно, но на других данных (например, если в помещении работают по субботам) такое правило может подвести.
- Эксперимент без освещённости показывает реалистичный сценарий для устройств мониторинга качества воздуха: по CO₂, температуре, влажности и их динамике занятость тоже определяется, но с заметно меньшей точностью.
- Применение: управление вентиляцией по фактической занятости, учёт использования помещений, энергосбережение.